<a href="https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I will create a feature vector for each content page using search performance, content characteristics, and freshness signals available before making a recommendation. I will exclude fields that reveal future outcomes or contain leakage.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_type",
    "main_intent"
]

X = df[features].copy()

# Fill numeric missing values
numeric_cols = X.select_dtypes(include="number").columns
X[numeric_cols] = X[numeric_cols].fillna(0)

# Fill categorical missing values
categorical_cols = X.select_dtypes(include="object").columns
X[categorical_cols] = X[categorical_cols].fillna("unknown")

# One-hot encode categorical variables
X = pd.get_dummies(X, columns=categorical_cols)

print("Feature matrix shape:", X.shape)
X.head()


Feature matrix shape: (30000, 19)


,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,10.0,0.67,2.05,3221.0,20457.0,187,20,3803,29,0.76,10.6,False,False,True,False,False,False,True,False
1,90.0,0.01,0.05,2481.0,15562.0,445,25,15320,7,0.05,20.3,False,False,True,False,True,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,141,20,12581,11,0.09,36.5,False,False,True,False,True,False,False,False
3,10.0,0.00,0.00,0.0,0.0,463,22,11751,58,0.49,6.2,False,False,True,True,False,False,False,False
4,0.0,0.00,0.00,2803.0,17469.0,263,14,19140,24,0.13,44.0,False,False,True,False,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

Feature meanings:

search_volume: estimated search demand for the topic.
competition: competition level around the query.
word_count and char_count: content size signals.
content_age_days: how old the page is.
days_since_last_update: freshness signal.
impressions_90d and clicks_90d: observed recent search performance.
ctr and avg_position: observed engagement and ranking signals.
Missing numerical values are filled with 0 and missing categorical values are marked as "unknown".

All features are available before the recommendation moment because they describe historical page information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Selected features:")
print(features)

print("\nMissing values before filling:")
print(df[features].isnull().sum())

print("\nFeature matrix shape:")
print(X.shape)


Selected features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_type', 'main_intent']

Missing values before filling:
search_volume             2468
competition               2468
cpc                       2468
word_count                7699
char_count                7699
content_age_days             0
days_since_last_update       0
impressions_90d              0
clicks_90d                   0
ctr                          0
avg_position                 0
content_type                 0
main_intent               2374
dtype: int64

Feature matrix shape:
(30000, 19)


## 3. The leakage hunt

I checked for columns that directly describe the outcome or contain future information. These fields should not be used because they would allow the model to see the answer before prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
possible_leakage = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

for col in possible_leakage:
    if col in df.columns:
        print("Excluded candidate:", col)
print("\nChecking if leakage columns are in the feature matrix:")

for col in possible_leakage:
    if col in X.columns:
        print(f"WARNING: {col} is still included!")
    else:
        print(f"OK: {col} is excluded.")


Excluded candidate: trend_direction
Excluded candidate: trend_pct
Excluded candidate: impressions_last_30d
Excluded candidate: clicks_last_30d
Excluded candidate: sessions_last_30d

Checking if leakage columns are in the feature matrix:
OK: trend_direction is excluded.
OK: trend_pct is excluded.
OK: impressions_last_30d is excluded.
OK: clicks_last_30d is excluded.
OK: sessions_last_30d is excluded.


## 4. What I excluded and why

Excluded fields:

trend_direction: target/proxy label, not an input feature.
trend_pct: directly describes the historical change and can leak the answer.
Future performance metrics: unavailable at prediction time.
Recommendation/action fields: created after analysis and would cause leakage.
client identifiers: not useful for generalization and may introduce privacy concerns.
These exclusions help ensure the model learns from available signals instead of memorizing answers.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = [
    "trend_direction",
    "trend_pct",
    "client_id"
]

print("Excluded fields:")
for field in excluded_features:
    print("-", field)

print("\nTotal excluded fields:", len(excluded_features))


Excluded fields:
- trend_direction
- trend_pct
- client_id

Total excluded fields: 3


## Self-check
* [x] Every section above is filled — markdown thinking AND the code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime → Run all)
* [x] No client names, URLs, or private queries anywhere
* [x] My claims use careful words: observed, measured, directional, decision-support
* [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Ahmed-coder874/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print(os.getcwd())


/content/flyrank-ml-internship


In [ ]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))


True
